In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from pathlib import Path
import torch
from dataclasses import dataclass

import os, json, re, time

os.environ["OPENAI_API_KEY"] = "..."

In [ ]:
QUESTION_TO_FULL_USER: dict[str,str] = {}

def get_model(model_name, is_finetuned: bool, using_websearch: bool):
    base = load_finetuned_model(model_name) if is_finetuned else load_base_model(model_name)

    def _run(question: str):
        # contexts: either [] (no web) or [full_user_content]
        contexts = []
        prompt = question
        if using_websearch:
            # Lấy lại full user content từ mapping (được nạp trong quá trình load samples) nếu có
            full_ctx = QUESTION_TO_FULL_USER.get(question)
            if full_ctx:
                contexts = [full_ctx]
                prompt = (
                    "Bạn là trợ lý tuyển sinh đại học. Dựa trên thông tin tham khảo dưới đây, trả lời câu hỏi một cách chính xác, ngắn gọn và trung thực.\n"
                    f"<THONG_TIN_THAM_KHAO>\n{full_ctx}\n</THONG_TIN_THAM_KHAO>\n\nCâu hỏi: {question}\nTrả lời:"\
                )
        answer = generate_with_model(base, prompt)
        return {"answer": answer, "contexts": contexts}
    return _run

# --- Internal caches ---
_MODEL_CACHE = {}

def load_finetuned_model(model_name):
    key = (model_name, "finetuned")
    if key in _MODEL_CACHE:
        return _MODEL_CACHE[key]
    try:
        base = load_base_model(model_name)  # returns dict with model+tokenizer
        model = base["model"]
        tokenizer = base["tokenizer"]
        adapter_dir = Path("finetune/qwen_lora_adapter") # unzip to this dir first
        if adapter_dir.exists():
            model = PeftModel.from_pretrained(model, str(adapter_dir))
            model.eval()
        _MODEL_CACHE[key] = {"model": model, "tokenizer": tokenizer}
        return _MODEL_CACHE[key]
    except Exception as e:  
        print(f"[ERROR] Finetuned load failed, fallback to base: {e}")
        return load_base_model(model_name)

def load_base_model(model_name):
    key = (model_name, "base")
    if key in _MODEL_CACHE:
        return _MODEL_CACHE[key]
    print(f"[LOAD] Base model {model_name}")
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="auto",
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        trust_remote_code=True
    )
    model.eval()
    _MODEL_CACHE[key] = {"model": model, "tokenizer": tokenizer}
    return _MODEL_CACHE[key]

def generate_with_model(bundle, prompt: str, max_new_tokens: int = 2048, temperature: float = 0.2):
    model = bundle["model"]
    tokenizer = bundle["tokenizer"]
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=4096).to(model.device)
    with torch.inference_mode():
        output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=temperature>0, temperature=temperature)
    text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    # Heuristic: strip prompt
    if prompt in text:
        text = text.split(prompt, 1)[-1].strip()
    return text.strip()

In [ ]:
DATA_PATH = Path('data/data_new.json')
OUTPUT_DIR = Path('eval_results'); OUTPUT_DIR.mkdir(exist_ok=True)

@dataclass
class Sample:
    question: str
    reference: str  # tham chiếu
    raw_user: str   # toàn bộ nội dung user

def load_raw_samples(path: Path) -> list[dict]:
    txt = path.read_text(encoding='utf-8')
    parts = re.split(r'\n}\s*{\n', txt)
    objs = []
    for i,p in enumerate(parts):
        if not p.strip():
            continue
        if not p.strip().startswith('{'): p = '{' + p
        if not p.strip().endswith('}'): p = p + '}'
        try:
            objs.append(json.loads(p))
        except Exception as e:
            print(f'[WARN] Bỏ qua block {i}: {e}')
    return objs


def extract_samples(objs: list[dict]) -> list[Sample]:
    out: list[Sample] = []
    for o in objs:
        msgs = o.get('messages', [])
        # Ensure roles (some file snippet trimmed system role fields) -> assume structure
        user_msg = None
        asst_msg = None
        for m in msgs:
            # guess by presence of 'Câu hỏi:' token for user part
            c = m.get('content','')
            if 'Câu hỏi:' in c and user_msg is None:
                user_msg = m
            elif user_msg is not None and asst_msg is None:
                asst_msg = m
        if not user_msg or not asst_msg:
            continue
        raw_user = user_msg.get('content','')
        q_match = re.findall(r'Câu hỏi:\s*(.+)$', raw_user, flags=re.MULTILINE)
        question = q_match[-1].strip() if q_match else raw_user[-256:]
        reference = asst_msg.get('content','').strip()
        if question and reference:
            out.append(Sample(question=question, reference=reference, raw_user=raw_user))
            QUESTION_TO_FULL_USER[question] = raw_user
    return out

raw_objs = load_raw_samples(DATA_PATH)
samples = extract_samples(raw_objs)
print(f'Loaded {len(samples)} samples')
if samples:
    print('Ví dụ câu hỏi:', samples[0].question[:120])

# ## Cấu hình mô hình cần đánh giá
BASE_MODEL_ID = os.getenv('BASE_MODEL_ID', 'Qwen/Qwen3-4B')
FINETUNE_MODEL_ID = BASE_MODEL_ID
CONFIGS = [
    ('pretrain_no_web', BASE_MODEL_ID, False, False),
    ('pretrain_web', BASE_MODEL_ID, False, True),
    ('finetune_no_web', FINETUNE_MODEL_ID, True, False),
    ('finetune_web', FINETUNE_MODEL_ID, True, True),
]

# ## Hàm sinh câu trả lời & cache
from functools import lru_cache

@lru_cache(maxsize=8)
def _get_runner(model_id: str, finetuned: bool, web: bool):
    return get_model(model_id, finetuned, web)

RESULT_CACHE_PATH = OUTPUT_DIR / 'answer_cache.json'
if RESULT_CACHE_PATH.exists():
    answer_cache = json.loads(RESULT_CACHE_PATH.read_text(encoding='utf-8'))
else:
    answer_cache = {}

def answer_sample(cfg_name: str, model_id: str, finetuned: bool, web: bool, question: str) -> dict:
    key = f'{cfg_name}:::{question[:200]}'
    if key in answer_cache:
        return answer_cache[key]
    runner = _get_runner(model_id, finetuned, web)
    out = runner(question)
    answer_cache[key] = out
    return out

# ## LLM Judge (GPTScore style)
try:
    from openai import OpenAI
    OPENAI_AVAILABLE = bool(os.getenv('OPENAI_API_KEY'))
except Exception:
    OPENAI_AVAILABLE = False

try:
    from sentence_transformers import SentenceTransformer
    import numpy as np
    _emb = SentenceTransformer('intfloat/multilingual-e5-small')
except Exception:
    _emb = None

JUDGE_MODEL = os.getenv('JUDGE_MODEL', 'gpt-4o-mini')

JUDGE_SYSTEM_PROMPT = (
    "Bạn là giám khảo đánh giá câu trả lời chatbot về tuyển sinh đại học. "
    "Trả về JSON với các trường: relevance (0.00-100.00), accuracy (0.00-100.00), faithfulness (0.00-100.00) (đơn vị: %)."
)

JUDGE_USER_TEMPLATE = (
    "Question: {question}\nReference Answer: {reference}\nModel Answer: {answer}\nContexts: {contexts}\n"
    "Định nghĩa:\n- Relevance: mức độ trả lời đúng trọng tâm câu hỏi.\n- Accuracy: tính đúng đắn thực tế so với reference (cho phép diễn đạt khác).\n- Faithfulness: có dựa vào contexts và không mâu thuẫn với contexts (nếu contexts rỗng, đánh giá dựa vào reference).\nTrả về JSON thuần."
)

def judge_llm(question: str, reference: str, answer: str, contexts: list[str]) -> dict:
    payload_user = JUDGE_USER_TEMPLATE.format(question=question, reference=reference, answer=answer, contexts='\n'.join(contexts))
    try:
        client = OpenAI()
        resp = client.chat.completions.create(
            model=JUDGE_MODEL,
            temperature=0.0,
            messages=[{"role":"system","content":JUDGE_SYSTEM_PROMPT}, {"role":"user","content":payload_user}]
        )
        txt = resp.choices[0].message.content.strip()
        m = re.search(r'{.*}', txt, flags=re.DOTALL)
        if m:
            import json as _json
            data = _json.loads(m.group(0))
            return {
                'relevance': float(data.get('relevance',0)),
                'accuracy': float(data.get('accuracy',0)),
                'faithfulness': float(data.get('faithfulness',0)),
            }
    except Exception as e:
        print('[WARN] LLM judge fail', e)


import pandas as pd

RESULTS = []
MAX_SAMPLES = int(os.getenv('EVAL_MAX', '100'))
subset = samples[:MAX_SAMPLES]
PROGRESS_EVERY = int(os.getenv('PROGRESS_EVERY', '1'))  # in samples
SAVE_EVERY = int(os.getenv('SAVE_EVERY', '5'))          # cache save interval

from tqdm import tqdm

start_time = time.time()
for cfg_name, model_id, finetuned, web in CONFIGS:
    print(f'>>> Evaluating {cfg_name} on {len(subset)} samples')
    sample_times = []
    iterator = tqdm(enumerate(subset, 1), total=len(subset), desc=cfg_name)
    for idx, s in iterator:
        t0 = time.time()
        ans_obj = answer_sample(cfg_name, model_id, finetuned, web, s.question)
        answer = ans_obj['answer']
        contexts = ans_obj.get('contexts', [])
        scores = judge_llm(s.question, s.reference, answer, contexts)
        RESULTS.append({
            'config': cfg_name,
            'question': s.question,
            'reference': s.reference,
            'answer': answer,
            'contexts_used': len(contexts),
            **scores
        })
        dt = time.time() - t0
        sample_times.append(dt)
        # progress feedback
        iterator.set_postfix({'last_s': f'{dt:.2f}', 'avg_s': f'{sum(sample_times)/len(sample_times):.2f}'})
        # periodic cache save
        if idx % SAVE_EVERY == 0:
            RESULT_CACHE_PATH.write_text(json.dumps(answer_cache, ensure_ascii=False, indent=2), encoding='utf-8')
    RESULT_CACHE_PATH.write_text(json.dumps(answer_cache, ensure_ascii=False, indent=2), encoding='utf-8')
    print(f'>>> {cfg_name} avg sample time: {sum(sample_times)/len(sample_times):.2f}s')

elapsed = time.time() - start_time
print(f'Done in {elapsed:.2f}s total')

df = pd.DataFrame(RESULTS)
summary = df.groupby('config')[['relevance','accuracy','faithfulness']].mean().reset_index()
print(summary)

SUMMARY_PATH = OUTPUT_DIR / 'summary.csv'
summary.to_csv(SUMMARY_PATH, index=False)
print('Saved:', SUMMARY_PATH)